In [1]:
# ============================================================
# FUNCTION 7 — WEEK 5 CLEAN, MEMORY-SAFE REBUILD
# ============================================================

import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel as C,
    Matern,
    WhiteKernel
)

# ------------------------------------------------------------
# 1. Load original Function 7 data
# ------------------------------------------------------------

X = np.load("function7/initial_inputs.npy")
Y = np.load("function7/initial_outputs.npy").reshape(-1)

# ------------------------------------------------------------
# 2. Add Weeks 1–4 exactly once
# ------------------------------------------------------------

week1_x = np.array([[
    0.148437,
    0.535927,
    0.276397,
    0.096809,
    0.345999,
    0.751013
]])

week1_y = np.array([
    1.1745584410367518
])

week2_x = np.array([[
    0.033169,
    0.329738,
    0.366358,
    0.234301,
    0.286315,
    0.704927
]])

week2_y = np.array([
    2.4608579737515917
])

week3_x = np.array([[
    0.000000,
    0.300093,
    0.304819,
    0.158057,
    0.252468,
    0.776583
]])

week3_y = np.array([
    1.8105475948256518
])

week4_x = np.array([[
    0.000000,
    0.321413,
    0.400365,
    0.258320,
    0.285038,
    0.762744
]])

week4_y = np.array([
    2.287474907540469
])

X = np.vstack([
    X,
    week1_x,
    week2_x,
    week3_x,
    week4_x
])

Y = np.concatenate([
    Y,
    week1_y,
    week2_y,
    week3_y,
    week4_y
])

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nBest observed input:", best_x)
print("Best observed output:", best_y)

# ------------------------------------------------------------
# 3. Fit an ARD Gaussian Process
# ------------------------------------------------------------

kernel = (
    C(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.full(6, 0.30),
        length_scale_bounds=(0.02, 3.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-2)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=7,
    random_state=57
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)

# ------------------------------------------------------------
# 4. Inspect fitted lengthscales
# ------------------------------------------------------------

lengthscales = np.asarray(
    gp.kernel_.k1.k2.length_scale
)

importance = 1 / lengthscales
importance = importance / importance.sum()

print("\nFitted lengthscales:", lengthscales)
print("Normalised input influence:")

for i, value in enumerate(importance, start=1):
    print(f"x{i}: {value:.4f}")

# ------------------------------------------------------------
# 5. Generate candidates
#
# Most candidates are centred on the Week 2 best.
# A smaller group searches between Weeks 2 and 4 because
# both points produced strong outputs.
# ------------------------------------------------------------

rng = np.random.default_rng(57)

very_local = rng.normal(
    loc=best_x,
    scale=[
        0.008,
        0.012,
        0.012,
        0.012,
        0.010,
        0.015
    ],
    size=(6_000, 6)
)

local = rng.normal(
    loc=best_x,
    scale=[
        0.025,
        0.035,
        0.035,
        0.035,
        0.030,
        0.040
    ],
    size=(8_000, 6)
)

wider_local = rng.normal(
    loc=best_x,
    scale=[
        0.055,
        0.070,
        0.070,
        0.070,
        0.060,
        0.080
    ],
    size=(5_000, 6)
)

# Search along the region between Week 2 and Week 4
blend_weights = rng.uniform(
    0,
    1,
    size=(4_000, 1)
)

blended_candidates = (
    blend_weights * week2_x
    + (1 - blend_weights) * week4_x
)

blended_candidates += rng.normal(
    loc=0,
    scale=[
        0.010,
        0.018,
        0.018,
        0.018,
        0.015,
        0.020
    ],
    size=(4_000, 6)
)

# Small global component because this is still six-dimensional
global_candidates = rng.uniform(
    0,
    1,
    size=(2_000, 6)
)

candidates = np.vstack([
    very_local,
    local,
    wider_local,
    blended_candidates,
    global_candidates
])

candidates = np.clip(candidates, 0, 1)

print("\nCandidates before filtering:", len(candidates))

# ------------------------------------------------------------
# 6. Remove candidates too close to previous observations
# ------------------------------------------------------------

tree = cKDTree(X)

minimum_distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    minimum_distance > 0.004
]

print("Candidates after filtering:", len(candidates))

# ------------------------------------------------------------
# 7. Predict in batches
# ------------------------------------------------------------

def predict_in_batches(model, points, batch_size=2000):

    means = []
    standard_deviations = []

    for start in range(0, len(points), batch_size):

        batch = points[
            start:start + batch_size
        ]

        batch_mean, batch_std = model.predict(
            batch,
            return_std=True
        )

        means.append(batch_mean)
        standard_deviations.append(batch_std)

    return (
        np.concatenate(means),
        np.concatenate(standard_deviations)
    )

mean, std = predict_in_batches(
    gp,
    candidates,
    batch_size=2000
)

# ------------------------------------------------------------
# 8. Expected Improvement
# ------------------------------------------------------------

xi = 0.005

improvement = mean - best_y - xi

with np.errstate(divide="ignore", invalid="ignore"):

    z = improvement / std

    expected_improvement = (
        improvement * norm.cdf(z)
        + std * norm.pdf(z)
    )

expected_improvement[std < 1e-12] = 0

# ------------------------------------------------------------
# 9. Upper Confidence Bound
# ------------------------------------------------------------

kappa = 0.55

ucb = mean + kappa * std

# ------------------------------------------------------------
# 10. Filter clearly weak predictions
# ------------------------------------------------------------

mean_filter = mean >= (best_y - 0.20)

if mean_filter.sum() < 100:
    mean_filter = mean >= np.percentile(mean, 90)

filtered_candidates = candidates[mean_filter]
filtered_mean = mean[mean_filter]
filtered_std = std[mean_filter]
filtered_ei = expected_improvement[mean_filter]
filtered_ucb = ucb[mean_filter]

print(
    "Candidates passing mean filter:",
    len(filtered_candidates)
)

# ------------------------------------------------------------
# 11. Combine EI and UCB
# ------------------------------------------------------------

ei_normalised = (
    filtered_ei - filtered_ei.min()
) / (
    np.ptp(filtered_ei) + 1e-12
)

ucb_normalised = (
    filtered_ucb - filtered_ucb.min()
) / (
    np.ptp(filtered_ucb) + 1e-12
)

# Week 2 remains the best, so favour exploitation
acquisition = (
    0.82 * ei_normalised
    + 0.18 * ucb_normalised
)

chosen_index = np.argmax(acquisition)

week5_query = filtered_candidates[
    chosen_index
]

# ------------------------------------------------------------
# 12. Print Week 5 proposal
# ------------------------------------------------------------

print("\nMethod:")
print("Local filtered Expected Improvement with small UCB component")

print("\nSuggested Week 5 query:")
print(week5_query)

print("\nPortal format:")
print("-".join(
    f"{value:.6f}"
    for value in week5_query
))

print(
    "\nPredicted mean:",
    filtered_mean[chosen_index]
)

print(
    "Predicted standard deviation:",
    filtered_std[chosen_index]
)

print(
    "Expected Improvement:",
    filtered_ei[chosen_index]
)

print(
    "UCB:",
    filtered_ucb[chosen_index]
)

print(
    "Distance from Week 2 best:",
    np.linalg.norm(
        week5_query - week2_x[0]
    )
)

print(
    "Distance from Week 4:",
    np.linalg.norm(
        week5_query - week4_x[0]
    )
)

X shape: (34, 6)
Y shape: (34,)

Best observed input: [0.033169 0.329738 0.366358 0.234301 0.286315 0.704927]
Best observed output: 2.4608579737515917

Fitted kernel:
0.704**2 * Matern(length_scale=[0.555, 3, 3, 0.173, 0.167, 0.459], nu=2.5) + WhiteKernel(noise_level=0.01)

Fitted lengthscales: [0.55524784 3.         3.         0.17318451 0.16698272 0.45893495]
Normalised input influence:
x1: 0.1098
x2: 0.0203
x3: 0.0203
x4: 0.3519
x5: 0.3650
x6: 0.1328

Candidates before filtering: 25000
Candidates after filtering: 25000
Candidates passing mean filter: 13824

Method:
Local filtered Expected Improvement with small UCB component

Suggested Week 5 query:
[0.         0.3380347  0.34666876 0.25036828 0.28318631 0.6253957 ]

Portal format:
0.000000-0.338035-0.346669-0.250368-0.283186-0.625396

Predicted mean: 2.3499430162446275
Predicted standard deviation: 0.14121040774129334
Expected Improvement: 0.016359109252729633
UCB: 2.4276087405023388
Distance from Week 2 best: 0.09027656563302966
D

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 3.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 3.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.01. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
